<a href="https://colab.research.google.com/github/yangrainie/TAICA_Generative-AI/blob/main/RAG_01_%E6%89%93%E9%80%A0%E5%90%91%E9%87%8F%E8%B3%87%E6%96%99%E5%BA%AB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG 1｜用 EmbeddingGemma 2 與 FAISS 建立中文 RAG 知識庫

步驟:<br>
1.上傳自己蒐集到的資料(.md檔)<br>
2.擷取RAG文字與metadata(取得文件正文、標題、來源網址、整理日期等資訊。)<br>
3.chunking(文件切割):將較長的文件切成多個小區塊，並保留必要的重疊內容。<br>
4.embedding(向量嵌入):使用 EmbeddingGemma 2 將每個 Chunk 轉換成數值向量。<br>
5.建立 FAISS 向量索引:將向量加入 FAISS 索引，建立後續搜尋所需的結構。<br>
6.保存與打包知識庫:保存 FAISS 索引、Chunk 文字、ID 對照及設定檔，最後壓縮成 ZIP 匯出。

準備教材：`rag-data.zip`。


## 1. 安裝套件

使用 Sentence Transformers 載入模型、NumPy 處理向量，以及 FAISS 搜尋向量。

安裝embedding、向量資料庫需要的套件

In [ ]:
%pip install -q "sentence-transformers==6.1.0" "transformers==5.20.1" "faiss-cpu==1.15.1"

## 2. 檢查環境與集中設定

In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import importlib.metadata as package_metadata
import hashlib
import json
import re
import tempfile
import zipfile

import torch
import faiss
from IPython.display import display
from sentence_transformers import SentenceTransformer
from huggingface_hub import HfApi

#### **設定embedding模型、批次處理大小、運算裝置，以及套件版本資訊。**

In [ ]:
MODEL_ID = "google/embeddinggemma-2" #指定要使用的 Embedding 模型
BATCH_SIZE = 4 #每批處理幾筆資料(調小可以降低記憶體使用量)
EXPECTED_DOCUMENT_COUNT = None  # 換自己的資料的大小(預期的文件數量，實際用途要看後續程式碼)
EXPORT_ZIP_NAME = "vector_db.zip" # 匯出檔案的名稱(可以自己改名稱)

# text 本身已含標題；這裡不再從 metadata 添加標題。
DOCUMENT_TEMPLATE = "title: none | text: {text}"
QUERY_TEMPLATE = "task: search result | query: {question}"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = (
    torch.bfloat16
    if DEVICE == "cuda" and torch.cuda.is_bf16_supported()
    else torch.float32
)
PACKAGE_NAMES = ["torch", "numpy", "sentence-transformers", "transformers", "faiss-cpu", "huggingface-hub"]
PACKAGE_VERSIONS = {name: package_metadata.version(name) for name in PACKAGE_NAMES}
print("裝置：", DEVICE, "／模型精度：", DTYPE)
if DEVICE == "cuda":
    print("GPU：", torch.cuda.get_device_name(0))
display(pd.DataFrame(PACKAGE_VERSIONS.items(), columns=["套件", "版本"]))

裝置： cuda ／模型精度： torch.bfloat16
GPU： Tesla T4


,套件,版本
0,torch,2.11.0+cu130
1,numpy,2.1.3
2,sentence-transformers,6.1.0
3,transformers,5.20.1
4,faiss-cpu,1.15.1
5,huggingface-hub,1.33.0


## 3. 上傳教材 ZIP

執行後選擇第一版教材 ZIP。程式只解開 TXT；README、CSV、問題集不會匯入知識庫。中文檔名由 Python 的 ZIP 工具讀取。

In [ ]:
from google.colab import files
from pathlib import Path
import tempfile

# 1.開啟上傳視窗，選擇檔案
uploaded = files.upload()

# 2.找出上傳檔案中副檔名為.md的檔案
md_names = [name for name in uploaded if name.lower().endswith(".md")]

# 3.確認有成功上傳文件
if not md_names:
    raise ValueError("找不到.md檔案，請確認上傳的文件。")

#archive_name = zip_names[0]
#archive_bytes = uploaded[archive_name]
#SOURCE_ARCHIVE_SHA256 = hashlib.sha256(archive_bytes).hexdigest()

# 4.建立暫存資料夾
DATA_DIR = Path(tempfile.mkdtemp(prefix="rag_documents_"))

# 5.將文件寫入暫存資料夾
for name in md_names:
    target = DATA_DIR / Path(name).name
    target.write_bytes(uploaded[name])

# 6. 建立 Markdown 文件路徑清單
md_paths = sorted(
    [p for p in DATA_DIR.rglob("*")
     if p.is_file() and p.suffix.lower() == ".md"],
    key=lambda p: p.relative_to(DATA_DIR).as_posix(),
)

# 7. 顯示上傳結果
print("Markdown 文件數量：", len(md_paths))

for path in md_paths:
    print("•", path.relative_to(DATA_DIR).as_posix())


Saving AppleWatchSE3.md to AppleWatchSE3.md
Saving COROSPACE4.md to COROSPACE4.md
Saving garmin_forerunner70.md to garmin_forerunner70.md
Saving garmin_venu3.md to garmin_venu3.md
Saving PolarPacer.md to PolarPacer.md
Markdown 文件數量： 5
• AppleWatchSE3.md
• COROSPACE4.md
• PolarPacer.md
• garmin_forerunner70.md
• garmin_venu3.md


## 4. 先看一篇原始文件

請觀察「標題」「來源網址」「教材整理日期」「【內容】」「【檢索練習問題】」等欄位。不是每個欄位都適合送進 embedding。

本教材只取標題與正文。若把文件裡的練習問題也送進去，搜尋可能直接命中題目，無法公平觀察正文的檢索效果。

可修改 `SAMPLE_INDEX`，換看其他文章。

In [ ]:
#指定要查看第幾份文件。Python 從 0 開始計算，所以 0 代表第一份。
SAMPLE_INDEX = 0

#從文件路徑清單中取出第一份文件的路徑。
sample_path = md_paths[SAMPLE_INDEX]
try:
    sample_raw = sample_path.read_text(encoding="utf-8-sig") #使用 UTF-8 編碼讀取文件內容
except UnicodeDecodeError as exc:
    raise ValueError(f"{sample_path.name} 不是可讀取的 UTF-8 文字。") from exc
print("檔名：", sample_path.name)
print("=" * 50)
print(sample_raw)

檔名： AppleWatchSE3.md
# Apple Watch SE 3

## 1. 基本資訊

- 品牌：Apple
- 型號：Apple Watch SE 3
- 推出年份：2025 年
- 產品類型：GPS 智慧手錶
- 官方售價：新台幣 7,900 元起（依錶殼尺寸與連線版本而異，購買前請確認官網價格）
- 錶殼材質：鋁金屬
- 錶殼顏色：午夜色、星光色
- 作業系統：watchOS
- 資料查詢日期：2026-10-28

## 2. 硬體規格

### 2.1 尺寸與重量

**40 公釐版本**
- 錶殼高度：40 公釐
- 錶殼寬度：34 公釐
- 錶殼厚度：10.7 公釐
- 螢幕解析度：324 × 394 像素
- 顯示面積：759 平方公釐
- GPS 版本重量：26.3 公克
- GPS + 行動網路版本重量：26.4 公克
- 適用手腕圍：130–200 公釐

**44 公釐版本**
- 錶殼高度：44 公釐
- 錶殼寬度：38 公釐
- 錶殼厚度：10.7 公釐
- 螢幕解析度：368 × 448 像素
- 顯示面積：977 平方公釐
- GPS 版本重量：32.9 公克
- GPS + 行動網路版本重量：33.0 公克
- 適用手腕圍：140–245 公釐

### 2.2 顯示器

- 顯示器類型：隨顯 Retina 顯示器，採用 OLED 與 LTPO 技術
- 峰值亮度：最高 1,000 尼特
- 最低亮度：2 尼特
- 像素密度：326 ppi
- 螢幕玻璃：Ion-X 玻璃錶面
- 耐用性：官方表示玻璃具備 4 倍抗裂能力

### 2.3 處理器與儲存空間

- 晶片：Apple S10
- 處理器：64 位元雙核心處理器
- 神經網路引擎：4 核心
- 儲存容量：64GB

### 2.4 感測器

- 第 2 代光學心率感測器
- 體溫感測器
- 指南針
- 常啟高度計
- 高 g 值加速度計
- 高動態範圍陀螺儀
- 環境光度感測器

## 3. GPS 與定位功能

- GPS：有
- 定位系統：L1 GPS、GLONASS、Galileo、QZSS、北斗
- 指南針：有
- 高度計：有，採常啟式高度計
- 健行定位：可使用指南針、座標、海拔高度、航點與 GPS 回溯等功能
- 路線功能：支援部分離線地圖及

## 5.第一個可修改函式: 取出 RAG 文字

比如你的文件以 `正文：` 開始，就修改擷取規則。若找不到正文，本函式明確報錯，不擅自把整份文件拿去 embedding。

這個回傳值會原樣保存；未來檢索後也將它交給 LLM，確保搜尋與回答使用同一份內容。

In [ ]:
#定義一個函式，接收原始文字，整理成 RAG 要使用的文字。
def extract_rag_text(raw_text):
    """從 Markdown 文件取得供 RAG 使用的文字。"""
    normalized = (
        raw_text.lstrip("\ufeff")
        .replace("\r\n", "\n")
        .replace("\r", "\n")
    )
    #清除首尾的空白
    body = normalized.strip()

    if not body:
        raise ValueError("文件內容為空，請確認 Markdown 檔案。")

    return body


sample_rag_text = extract_rag_text(sample_raw)

print(sample_rag_text)
print("\n擷取後字元數：", len(sample_rag_text))


# Apple Watch SE 3

## 1. 基本資訊

- 品牌：Apple
- 型號：Apple Watch SE 3
- 推出年份：2025 年
- 產品類型：GPS 智慧手錶
- 官方售價：新台幣 7,900 元起（依錶殼尺寸與連線版本而異，購買前請確認官網價格）
- 錶殼材質：鋁金屬
- 錶殼顏色：午夜色、星光色
- 作業系統：watchOS
- 資料查詢日期：2026-10-28

## 2. 硬體規格

### 2.1 尺寸與重量

**40 公釐版本**
- 錶殼高度：40 公釐
- 錶殼寬度：34 公釐
- 錶殼厚度：10.7 公釐
- 螢幕解析度：324 × 394 像素
- 顯示面積：759 平方公釐
- GPS 版本重量：26.3 公克
- GPS + 行動網路版本重量：26.4 公克
- 適用手腕圍：130–200 公釐

**44 公釐版本**
- 錶殼高度：44 公釐
- 錶殼寬度：38 公釐
- 錶殼厚度：10.7 公釐
- 螢幕解析度：368 × 448 像素
- 顯示面積：977 平方公釐
- GPS 版本重量：32.9 公克
- GPS + 行動網路版本重量：33.0 公克
- 適用手腕圍：140–245 公釐

### 2.2 顯示器

- 顯示器類型：隨顯 Retina 顯示器，採用 OLED 與 LTPO 技術
- 峰值亮度：最高 1,000 尼特
- 最低亮度：2 尼特
- 像素密度：326 ppi
- 螢幕玻璃：Ion-X 玻璃錶面
- 耐用性：官方表示玻璃具備 4 倍抗裂能力

### 2.3 處理器與儲存空間

- 晶片：Apple S10
- 處理器：64 位元雙核心處理器
- 神經網路引擎：4 核心
- 儲存容量：64GB

### 2.4 感測器

- 第 2 代光學心率感測器
- 體溫感測器
- 指南針
- 常啟高度計
- 高 g 值加速度計
- 高動態範圍陀螺儀
- 環境光度感測器

## 3. GPS 與定位功能

- GPS：有
- 定位系統：L1 GPS、GLONASS、Galileo、QZSS、北斗
- 指南針：有
- 高度計：有，採常啟式高度計
- 健行定位：可使用指南針、座標、海拔高度、航點與 GPS 回溯等功能
- 路線功能：支援部分離線地圖及自訂路線功能，實際可用範圍依功能與地區而異

## 6. 第二個可修改函式：取出 metadata

`extract_metadata(raw_text, filename)` 回傳標題、網址、日期、檔名和文件性質。metadata 是追溯資料用的資訊，不會整包送進 embedding。

**換自己的資料，可在這裡增減欄位。** 例如食譜可以增加料理類型，課程介紹可以增加學期。來源與日期找不到時用 `None`，不自行捏造；標題缺失時使用檔名。

這裡的 `date` 是教材整理日期，不是法規生效日期。保留「教學改寫摘要」的性質，避免後續展示誤稱官方原文。

In [ ]:

def extract_metadata(raw_text, filename):
    """從 Markdown 文件取得描述資訊。"""
    normalized = (
        raw_text.lstrip("\ufeff")
        .replace("\r\n", "\n")
        .replace("\r", "\n")
    )

    def read_field(names):
        for name in names:
            match = re.search(
                rf"^\s*(?:[-*]\s*)?{re.escape(name)}[：:][ \t]*(.*)$",
                normalized,
                flags=re.MULTILINE
            )
            if match and match.group(1).strip():
                return match.group(1).strip()
        return None

    # 讀取第一個一級標題
    title_match = re.search(
        r"^#\s+(.+)$",
        normalized,
        flags=re.MULTILINE
    )
    title = (
        title_match.group(1).strip()
        if title_match
        else Path(filename).stem
    )

    # 讀取文件中第一個網址
    url_match = re.search(
        r"https?://[^\s)\]]+",
        normalized
    )
    source_url = url_match.group(0) if url_match else None

    # 讀取資料查詢日期
    date = read_field([
        "資料查詢日期",
        "教材整理日期",
        "資料整理日期",
        "整理日期"
    ])

    return {
        "title": title,
        "source_url": source_url,
        "date": date,
        "filename": filename,
        "document_type": read_field(["資料類型"]) or "運動手錶產品資料",
    }


sample_metadata = extract_metadata(
    sample_raw,
    sample_path.relative_to(DATA_DIR).as_posix()
)

print(json.dumps(sample_metadata, ensure_ascii=False, indent=2))


{
  "title": "Apple Watch SE 3",
  "source_url": "https://www.apple.com/tw/apple-watch-se-3/",
  "date": "2026-10-28",
  "filename": "AppleWatchSE3.md",
  "document_type": "運動手錶產品資料"
}


## 7. 批次整理成統一格式

把兩個函式組合起來，每篇得到 `doc_id`、`text` 和 `metadata`。檔案先排序，再按順序給編號；只要教材及排序不變，編號便相同。新增文件可能改變編號，因此需要整套重建索引與對照。

請檢查列表中每篇是否有標題、正文及來源。缺少 metadata 會提醒；正文缺失則停止並指出檔名，讓你回到前兩格修改函式。

**將上傳的Markdown運動手錶資料，逐一讀取、擷取正文與metadata，再整理成統一的資料結構，讓後續的embedding和FAISS可以使用。**

**embedding(向量嵌入):是一種將文字、圖片、聲音或商品等資料轉換成一串實數數字（向量）的技術。**

In [ ]:
documents = []
for number, path in enumerate(md_paths, start=1):
    filename = path.relative_to(DATA_DIR).as_posix()
    try:
        raw_text = path.read_text(encoding="utf-8-sig")
        if "\ufffd" in raw_text:
            raise ValueError("文字含有替代字元，請檢查原始編碼。")
        rag_text = extract_rag_text(raw_text)
        metadata = extract_metadata(raw_text, filename)
    except (ValueError, UnicodeDecodeError) as exc:
        raise ValueError(f"處理失敗：{filename}；原因：{exc}") from exc
    documents.append({"doc_id": f"doc_{number:03d}", "text": rag_text, "metadata": metadata})

if not documents:
    raise ValueError("沒有可建立索引的文件。")
index_doc_ids = [doc["doc_id"] for doc in documents]
if len(set(index_doc_ids)) != len(index_doc_ids):
    raise ValueError("doc_id 重複，請檢查編號規則。")

summary = pd.DataFrame([
    {"doc_id": doc["doc_id"], "標題": doc["metadata"]["title"],
     "正文含標題字元數": len(doc["text"]), "來源": doc["metadata"]["source_url"],
     "整理日期": doc["metadata"]["date"]}
    for doc in documents
])
display(summary)
for doc in documents:
    missing = [key for key in ("source_url", "date") if not doc["metadata"].get(key)]
    if missing:
        print("提醒：", doc["doc_id"], "缺少", ", ".join(missing))
print("第一筆統一格式：")
print(json.dumps(documents[0], ensure_ascii=False, indent=2))

,doc_id,標題,正文含標題字元數,來源,整理日期
0,doc_001,Apple Watch SE 3,4242,https://www.apple.com/tw/apple-watch-se-3/,2026-10-28
1,doc_002,COROS PACE 4,4627,https://www.corostw.com/products/pace4-aluminum,2026-10-28
2,doc_003,Polar Pacer,4068,https://www.polar.com/tw-zh/pacer,2026-10-28
3,doc_004,Garmin Forerunner70,3716,https://www.garmin.com.tw/products/wearables/f...,2026-10-28
4,doc_005,Garmin Venu 3,2978,https://www.garmin.com.tw/products/wearables/v...,2026-10-28


第一筆統一格式：
{
  "doc_id": "doc_001",
  "text": "# Apple Watch SE 3\n\n## 1. 基本資訊\n\n- 品牌：Apple\n- 型號：Apple Watch SE 3\n- 推出年份：2025 年\n- 產品類型：GPS 智慧手錶\n- 官方售價：新台幣 7,900 元起（依錶殼尺寸與連線版本而異，購買前請確認官網價格）\n- 錶殼材質：鋁金屬\n- 錶殼顏色：午夜色、星光色\n- 作業系統：watchOS\n- 資料查詢日期：2026-10-28\n\n## 2. 硬體規格\n\n### 2.1 尺寸與重量\n\n**40 公釐版本**\n- 錶殼高度：40 公釐\n- 錶殼寬度：34 公釐\n- 錶殼厚度：10.7 公釐\n- 螢幕解析度：324 × 394 像素\n- 顯示面積：759 平方公釐\n- GPS 版本重量：26.3 公克\n- GPS + 行動網路版本重量：26.4 公克\n- 適用手腕圍：130–200 公釐\n\n**44 公釐版本**\n- 錶殼高度：44 公釐\n- 錶殼寬度：38 公釐\n- 錶殼厚度：10.7 公釐\n- 螢幕解析度：368 × 448 像素\n- 顯示面積：977 平方公釐\n- GPS 版本重量：32.9 公克\n- GPS + 行動網路版本重量：33.0 公克\n- 適用手腕圍：140–245 公釐\n\n### 2.2 顯示器\n\n- 顯示器類型：隨顯 Retina 顯示器，採用 OLED 與 LTPO 技術\n- 峰值亮度：最高 1,000 尼特\n- 最低亮度：2 尼特\n- 像素密度：326 ppi\n- 螢幕玻璃：Ion-X 玻璃錶面\n- 耐用性：官方表示玻璃具備 4 倍抗裂能力\n\n### 2.3 處理器與儲存空間\n\n- 晶片：Apple S10\n- 處理器：64 位元雙核心處理器\n- 神經網路引擎：4 核心\n- 儲存容量：64GB\n\n### 2.4 感測器\n\n- 第 2 代光學心率感測器\n- 體溫感測器\n- 指南針\n- 常啟高度計\n- 高 g 值加速度計\n- 高動態範圍陀螺儀\n- 環境光度感測器\n\n## 3. GPS 與定位功能\n\n- GPS：有\n- 定位系統：L1 GPS、GLONASS、G

## 8. 載入 EmbeddingGemma 2

先查詢模型目前的確切版本（commit），再載入該版本，將版本記錄到輸出設定，方便第二份 Colab 使用相同模型。只啟用文字部分，使用適合裝置的精度。

首次下載需要等待。本步使用公開模型，不讀取 Groq 或其他密鑰。若下載失敗，先確認網路；模型架構不受支援時，確認安裝格已完成，並重新啟動工作階段。程式不會默默改用別的模型。

In [ ]:
try:
    MODEL_REVISION = HfApi().model_info(MODEL_ID, token=False).sha
    model = SentenceTransformer(
        MODEL_ID,
        revision=MODEL_REVISION,
        token=False,
        device=DEVICE,
        config_kwargs={"vision_config": None, "audio_config": None},
        model_kwargs={"dtype": DTYPE, "attn_implementation": "eager"},
    )
except Exception as exc:
    raise RuntimeError(
        "EmbeddingGemma 2 載入失敗。請確認網路、套件版本，必要時重新啟動工作階段；"
        "若 GPU 記憶體不足，改用 CPU。未替換成其他模型。"
    ) from exc

print("模型：", MODEL_ID)
print("版本：", MODEL_REVISION)
print("裝置：", model.device)
print("模型精度：", model.dtype)
print("向量維度：", model.get_sentence_embedding_dimension())
print("模型單筆 token 上限：", model.max_seq_length)

modules.json:   0%|          | 0.00/413 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/1.56k [00:00<?, ?B/s]

README.md:   0%|          | 0.00/22.2k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.46k [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/97.0 [00:00<?, ?B/s]

模型： google/embeddinggemma-2
版本： f9972ac22e696a105c482202646979e33e4b5a1e
裝置： cuda:0
模型精度： torch.bfloat16
向量維度： 768
模型單筆 token 上限： 1000000000000000019884624838656


/tmp/ipykernel_677/465943176.py:21: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("向量維度：", model.get_sentence_embedding_dimension())


## 9. 檢查輸入長度，避免靜默截斷

本模型採用不同的任務前綴來區分文件和問題。文件本身已有標題，所以組成 `title: none | text: {text}`；問題則用 `task: search result | query: {question}`。

下格計算實際送入模型的文件文字（含前綴）長度。超過上限便停止，不會只取前半段。換成長文件時，可以在後續課程加入 chunking，或重新設計文件單位。

**由於第一次執行時，發現上傳的五個檔案Token數超過設定上限，GPT建議在每份產品說明加入文字切分(chunking)，拆成較小的區塊，再讓每個區塊分別產生向量(embedding)。**

In [ ]:

# 1. 設定文字切分參數
CHUNK_SIZE = 1800
CHUNK_OVERLAP = 100 #在切分區塊時，單獨閱讀第二段，可能缺少前面的主詞或條件，造成上下文斷裂的問題，為了避免故加上重疊。

# 2. 檢查切分參數
if CHUNK_SIZE <= CHUNK_OVERLAP or CHUNK_OVERLAP < 0:
    raise ValueError("切分大小必須大於重疊大小，且重疊大小不可小於 0。")

# 3. 將每份完整文件切分成多個區塊
chunked_documents = []

for doc in documents:
    text = doc["text"]

    # 先將文字轉成 Token ID，不加入特殊 Token
    token_ids = model.tokenizer.encode(
        text,
        add_special_tokens=False
    )

    start = 0
    chunk_number = 1

    while start < len(token_ids):
        end = min(start + CHUNK_SIZE, len(token_ids))

        # 將這一段 Token 還原成文字
        chunk_text = model.tokenizer.decode(
            token_ids[start:end],
            skip_special_tokens=True
        )

        # 每個區塊保留產品標題，增加檢索時的上下文
        title = doc["metadata"]["title"]
        chunk_text = f"產品名稱：{title}\n\n{chunk_text}"

        chunked_documents.append({
            "doc_id": f"{doc['doc_id']}_chunk_{chunk_number:03d}",
            "text": chunk_text,
            "metadata": {
                **doc["metadata"],
                "chunk_number": chunk_number,
                "source_doc_id": doc["doc_id"]
            }
        })

        chunk_number += 1

        if end == len(token_ids):
            break

        # 保留部分前一區塊的內容，減少切分造成的上下文斷裂
        start = end - CHUNK_OVERLAP

# 4. 以切分後的文件取代原本的文件清單
documents = chunked_documents

# 5. 重新建立文件 ID 清單
index_doc_ids = [doc["doc_id"] for doc in documents]

print("原始文件數：5")
print("切分後的區塊數：", len(documents))
print("第一個區塊：")
print(documents[0]["text"][:500])


原始文件數：5
切分後的區塊數： 10
第一個區塊：
產品名稱：Apple Watch SE 3

# Apple Watch SE 3

## 1. 基本資訊

- 品牌：Apple
- 型號：Apple Watch SE 3
- 推出年份：2025 年
- 產品類型：GPS 智慧手錶
- 官方售價：新台幣 7,900 元起（依錶殼尺寸與連線版本而異，購買前請確認官網價格）
- 錶殼材質：鋁金屬
- 錶殼顏色：午夜色、星光色
- 作業系統：watchOS
- 資料查詢日期：2026-10-28

## 2. 硬體規格

### 2.1 尺寸與重量

**40 公釐版本**
- 錶殼高度：40 公釐
- 錶殼寬度：34 公釐
- 錶殼厚度：10.7 公釐
- 螢幕解析度：324 × 394 像素
- 顯示面積：759 平方公釐
- GPS 版本重量：26.3 公克
- GPS + 行動網路版本重量：26.4 公克
- 適用手腕圍：130–200 公釐

**44 公釐版本**
- 錶殼高度：44 公釐
- 錶殼寬度：38 公釐
- 錶殼厚度：10.7 公釐
- 螢幕解析度：368 × 448 像素
- 顯示面積：977 平方公釐
- GP


In [ ]:
document_inputs = [DOCUMENT_TEMPLATE.format(text=doc["text"]) for doc in documents]
TOKEN_LIMIT = min(2048, int(model.max_seq_length))

if TOKEN_LIMIT <= 0:
    raise ValueError("輸入長度上限必須大於0。")

def count_input_tokens(text):
    return len(model.tokenizer(text, add_special_tokens=True, truncation=False)["input_ids"])

token_counts = [count_input_tokens(text) for text in document_inputs]
length_table = pd.DataFrame({"doc_id": index_doc_ids, "tokens_含前綴": token_counts})
display(length_table)
long_docs = [doc_id for doc_id, length in zip(index_doc_ids, token_counts) if length > TOKEN_LIMIT]
if long_docs:
    raise ValueError(f"以下文件超過 {TOKEN_LIMIT} tokens，請先調整資料：{long_docs}")
print("本教材使用的 token 上限：", TOKEN_LIMIT)
print("全部文件均未超過上限；不需要截斷。")
print("第一筆真正送入 embedding 的文字：\n", document_inputs[0])





,doc_id,tokens_含前綴
0,doc_001_chunk_001,1818
1,doc_001_chunk_002,1190
2,doc_002_chunk_001,1818
3,doc_002_chunk_002,1403
4,doc_003_chunk_001,1816
5,doc_003_chunk_002,775
6,doc_004_chunk_001,1819
7,doc_004_chunk_002,978
8,doc_005_chunk_001,1819
9,doc_005_chunk_002,445


本教材使用的 token 上限： 2048
全部文件均未超過上限；不需要截斷。
第一筆真正送入 embedding 的文字：
 title: none | text: 產品名稱：Apple Watch SE 3

# Apple Watch SE 3

## 1. 基本資訊

- 品牌：Apple
- 型號：Apple Watch SE 3
- 推出年份：2025 年
- 產品類型：GPS 智慧手錶
- 官方售價：新台幣 7,900 元起（依錶殼尺寸與連線版本而異，購買前請確認官網價格）
- 錶殼材質：鋁金屬
- 錶殼顏色：午夜色、星光色
- 作業系統：watchOS
- 資料查詢日期：2026-10-28

## 2. 硬體規格

### 2.1 尺寸與重量

**40 公釐版本**
- 錶殼高度：40 公釐
- 錶殼寬度：34 公釐
- 錶殼厚度：10.7 公釐
- 螢幕解析度：324 × 394 像素
- 顯示面積：759 平方公釐
- GPS 版本重量：26.3 公克
- GPS + 行動網路版本重量：26.4 公克
- 適用手腕圍：130–200 公釐

**44 公釐版本**
- 錶殼高度：44 公釐
- 錶殼寬度：38 公釐
- 錶殼厚度：10.7 公釐
- 螢幕解析度：368 × 448 像素
- 顯示面積：977 平方公釐
- GPS 版本重量：32.9 公克
- GPS + 行動網路版本重量：33.0 公克
- 適用手腕圍：140–245 公釐

### 2.2 顯示器

- 顯示器類型：隨顯 Retina 顯示器，採用 OLED 與 LTPO 技術
- 峰值亮度：最高 1,000 尼特
- 最低亮度：2 尼特
- 像素密度：326 ppi
- 螢幕玻璃：Ion-X 玻璃錶面
- 耐用性：官方表示玻璃具備 4 倍抗裂能力

### 2.3 處理器與儲存空間

- 晶片：Apple S10
- 處理器：64 位元雙核心處理器
- 神經網路引擎：4 核心
- 儲存容量：64GB

### 2.4 感測器

- 第 2 代光學心率感測器
- 體溫感測器
- 指南針
- 常啟高度計
- 高 g 值加速度計
- 高動態範圍陀螺儀
- 環境光度感測器

## 3. GPS 與定位功能

- GPS：有
- 定位系統：L1 GPS、GLONASS、Galile

## 10. 產生並觀察文件向量

一篇文件對應矩陣中的一列，欄數是向量維度。`prompt=""` 表示前綴已自行加入，不讓模型額外再加一次。

向量正規化後長度約為1，內積便等於 cosine similarity。下格也將資料轉為 FAISS 使用的 float32 連續陣列，檢查空向量、非有限值及形狀。

**將前面切分好的文件區塊轉換成數值向量（Embedding），並檢查向量是否正常。**

In [ ]:
embeddings = model.encode(
    document_inputs,
    prompt="",
    batch_size=BATCH_SIZE, #每批處理幾筆文字
    convert_to_numpy=True, #將結果轉成NumPy陣列
    normalize_embeddings=True, #將輸出的向量正規化
    show_progress_bar=True,
)
embeddings = np.ascontiguousarray(embeddings, dtype=np.float32)
if embeddings.ndim != 2 or embeddings.shape[0] != len(documents):
    raise ValueError("向量形狀與文件數不符。")
if not np.isfinite(embeddings).all():
    raise ValueError("向量含有 NaN 或無限值。")
if np.any(np.linalg.norm(embeddings, axis=1) <= 0):
    raise ValueError("出現零向量。")
faiss.normalize_L2(embeddings)
DIMENSION = int(embeddings.shape[1])
if DIMENSION != int(model.get_sentence_embedding_dimension()):
    raise ValueError("實際向量維度與模型宣告不符。")
print("矩陣形狀（文件數, 向量維度）：", embeddings.shape)
print("第一篇向量的前10個數字：", embeddings[0, :10])
print("前5篇向量長度：", np.linalg.norm(embeddings[:5], axis=1))

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

矩陣形狀（文件數, 向量維度）： (10, 768)
第一篇向量的前10個數字： [ 0.03293969 -0.00323297 -0.04928754 -0.00469696 -0.01006491  0.04465158
  0.01421287 -0.03611166 -0.00253148 -0.12785481]
前5篇向量長度： [1.        1.0000001 1.        1.        1.       ]


/tmp/ipykernel_677/119107976.py:18: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  if DIMENSION != int(model.get_sentence_embedding_dimension()):


## 11. 建立 FAISS 精確搜尋索引

`IndexFlatIP` 比較問題向量與所有文件向量的內積。因為雙方都正規化，排名就是 cosine similarity 的排名。

FAISS 回傳的是向量編號，**不是原文**。`index_doc_ids` 保存向量順序與文件編號的對應，再透過 `doc_id` 找回正文與 metadata。

In [ ]:
index = faiss.IndexFlatIP(DIMENSION)
index.add(embeddings)
documents_by_id = {doc["doc_id"]: doc for doc in documents}
assert index.ntotal == len(index_doc_ids) == len(documents)
print("索引類型：", type(index).__name__)
print("索引向量數：", index.ntotal)
print("索引維度：", index.d)
display(pd.DataFrame({"FAISS編號": range(len(index_doc_ids)), "doc_id": index_doc_ids}))

索引類型： IndexFlatIP
索引向量數： 10
索引維度： 768


,FAISS編號,doc_id
0,0,doc_001_chunk_001
1,1,doc_001_chunk_002
2,2,doc_002_chunk_001
3,3,doc_002_chunk_002
4,4,doc_003_chunk_001
5,5,doc_003_chunk_002
6,6,doc_004_chunk_001
7,7,doc_004_chunk_002
8,8,doc_005_chunk_001
9,9,doc_005_chunk_002


## 12. 定義問題向量化與搜尋函式

問題也要用同一個 embedding 模型向量化，但使用查詢專用前綴。`top_k` 表示取回幾篇文件；程式限制它不超過實際文件數。

搜尋函式只做檢索，還沒有呼叫 LLM。精確搜尋保證依目前向量找到分數最高者，不保證文件一定足以回答問題。

In [ ]:
#將user的問題轉成向量
def embed_question(question):
    if not isinstance(question, str) or not question.strip():
        raise ValueError("請輸入非空白問題。")
    query_input = QUERY_TEMPLATE.format(question=question.strip())
    if count_input_tokens(query_input) > TOKEN_LIMIT:
        raise ValueError("問題超過模型輸入長度上限，請縮短問題。")
    query_vector = model.encode(
        [query_input], prompt="", convert_to_numpy=True,
        normalize_embeddings=True, show_progress_bar=False,
    )
    query_vector = np.ascontiguousarray(query_vector, dtype=np.float32)
    if query_vector.shape != (1, DIMENSION) or not np.isfinite(query_vector).all():
        raise ValueError("問題向量的維度或數值不正確。")
    if np.linalg.norm(query_vector) <= 0:
        raise ValueError("問題產生零向量。")
    #確認向量維度正確後進行正規化
    faiss.normalize_L2(query_vector)
    return query_vector

#用問題向量搜尋最相關的文件區塊
#top_k=n，表示預設取出n個最相關的文件區塊
def search_documents(question, top_k=1):
    if isinstance(top_k, bool) or not isinstance(top_k, (int, np.integer)) or top_k < 1:
        raise ValueError("top_k 必須是大於0的整數。")
    query_vector = embed_question(question)
    scores, positions = index.search(query_vector, min(int(top_k), index.ntotal))
    results = []
    for rank, (position, score) in enumerate(zip(positions[0], scores[0]), start=1):
        doc_id = index_doc_ids[int(position)]
        results.append({"rank": rank, "score": float(score), **documents_by_id[doc_id]})
    return results

## 13. 試查一次，閱讀真正取回的內容

修改 `QUESTION` 和 `TOP_K`，重新執行本格即可查詢。先看標題及分數，再讀完整正文，判斷是否有足夠依據。

可以比較 `top_k=1`、`3`、`5`。。**相似度不是正確率，也不是模型有答案的機率。**

In [ ]:
QUESTION = "哪個手錶的電池續航力最好?"
TOP_K = 1
results = search_documents(QUESTION, TOP_K)
display(pd.DataFrame([
    {"排名": r["rank"], "doc_id": r["doc_id"], "標題": r["metadata"]["title"],
     "cosine_similarity": r["score"], "來源": r["metadata"]["source_url"]}
    for r in results
]))
for result in results:
    print(f"\n[{result['doc_id']}] {result['metadata']['title']} ({result['score']:.4f})")
    print(result["text"])
    print("來源：", result["metadata"]["source_url"] or "未提供")

,排名,doc_id,標題,cosine_similarity,來源
0,1,doc_003_chunk_001,Polar Pacer,0.724452,https://www.polar.com/tw-zh/pacer



[doc_003_chunk_001] Polar Pacer (0.7245)
產品名稱：Polar Pacer

# Polar Pacer 


## 1. 基本資料

- **產品名稱：** Polar Pacer
- **品牌：** Polar
- **產品類型：** GPS 運動手錶
- **產品定位：** 以跑步與日常運動訓練為主，提供 GPS 定位、手腕式心率追蹤、訓練分析、睡眠與恢復功能。
- **台灣官方產品頁標示價格：** NT$7,700；價格可能變動。
- **錶殼尺寸：** 45 × 45 × 11.5 mm
- **重量：** 含腕帶 40 g；不含腕帶 23 g
- **防水等級：** WR50
- **官方產品頁展示顏色：** Night Black（夜黑色）
-   資料查詢日期：2026-10-28
- **官方產品頁：** https://www.polar.com/tw-zh/pacer

## 2. 硬體與技術規格

### 2.1 螢幕與外觀

- **螢幕類型：** MIP（Memory-in-Pixel）顯示器
- **螢幕尺寸：** 1.2 吋
- **解析度：** 240 × 240 像素
- **常時顯示：** 支援
- **螢幕材質：** Gorilla Glass
- **錶殼材質：** 塑膠
- **操作溫度：** -10°C 至 +50°C
- **防水：** WR50；請依官方防水使用說明操作，不應直接視為可用於所有水上活動。

### 2.2 處理效能與儲存空間

- **CPU 速度：** 192 MHz
- **記憶體：** 5 MB
- **儲存空間：** 32 MB

### 2.3 定位與連線

- **內建定位：** 有
- **支援衛星定位系統：** GPS、GLONASS、Galileo、QZSS
- **輔助定位：** A-GPS、A-GLONASS、A-Galileo、A-QZSS
- **藍牙：** Bluetooth Low Energy（低功耗藍牙）
- **充電／資料傳輸線：** 專用 USB-A 連接線
- **手機通知：** 支援推送通知
- **音樂控制：** 支援控制連線手機上的音樂；不代表手錶可獨立儲存或播放音樂。
- **天氣：** 支援天氣資訊功能
- *

## 14. 保存索引、文件、編號對照與設定

第二份 Colab 需要下面四個檔案，不能只拿 FAISS 索引：

| 檔案 | 用途 |
|---|---|
| `index.faiss` | 文件向量與搜尋索引 |
| `documents.json` | 取回後提供給 LLM 的正文與來源資訊 |
| `index_doc_ids.json` | FAISS 編號對應的文件編號 |
| `config.json` | 模型確切版本、前綴、維度、正規化、套件版本及檢查資訊 |

不保存模型權重、API 金鑰或其他環境變數。第二份 Colab 載入相同模型版本，只重新計算問題向量。

In [ ]:

# 1. 建立保存資料的資料夾
EXPORT_DIR = Path(tempfile.mkdtemp(prefix="rag_index_"))

# 2. 保存 FAISS 索引
faiss.write_index(index, str(EXPORT_DIR / "index.faiss"))

# 3. 保存 Chunk 文字與索引編號對照
for filename, data in [
    ("documents.json", documents),
    ("index_doc_ids.json", index_doc_ids),
]:
    (EXPORT_DIR / filename).write_text(
        json.dumps(data, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

# 4. 保存知識庫設定
config = {
    "format_version": "1.0",
    "created_at": datetime.now(
        ZoneInfo("Asia/Taipei")
    ).isoformat(timespec="seconds"),

    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "model_config_kwargs": {
        "vision_config": None,
        "audio_config": None
    },

    "dimension": DIMENSION,
    "normalize_embeddings": True,
    "index_type": "IndexFlatIP",
    "metric": "cosine_similarity_via_normalized_inner_product",

    "document_template": DOCUMENT_TEMPLATE,
    "query_template": QUERY_TEMPLATE,
    "encode_prompt": "",
    "max_seq_length": TOKEN_LIMIT,

    # 目前是一個 Chunk 對應一個向量
    "document_count": len(documents),
    "document_unit": "one_chunk_one_vector",
    "document_text_includes_title": True,

    "package_versions": PACKAGE_VERSIONS,
    "build_device": DEVICE,
    "build_dtype": str(DTYPE),

    # 記錄原始文件數量及來源檔名
    "source_document_count": len({
        doc["metadata"].get("source_doc_id", doc["doc_id"])
        for doc in documents
    }),
    "source_files": sorted({
        doc["metadata"]["filename"]
        for doc in documents
        if doc["metadata"].get("filename")
    }),

    # 計算已保存檔案的 SHA-256
    "file_sha256": {
        name: hashlib.sha256(
            (EXPORT_DIR / name).read_bytes()
        ).hexdigest()
        for name in [
            "index.faiss",
            "documents.json",
            "index_doc_ids.json"
        ]
    },
}

(EXPORT_DIR / "config.json").write_text(
    json.dumps(config, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

# 5. 顯示保存結果
print("知識庫已保存至：", EXPORT_DIR)
print("\n已保存檔案：")

for path in sorted(EXPORT_DIR.iterdir()):
    print(f"{path.name} ({path.stat().st_size:,} bytes)")

print("\n文件區塊數：", len(documents))
print("索引向量數：", index.ntotal)
print("向量維度：", DIMENSION)
print("模型版本及前綴已寫入 config.json。")


知識庫已保存至： /tmp/rag_index_xt42e_k0

已保存檔案：
config.json (1,431 bytes)
documents.json (48,376 bytes)
index.faiss (30,765 bytes)
index_doc_ids.json (232 bytes)

文件區塊數： 10
索引向量數： 10
向量維度： 768
模型版本及前綴已寫入 config.json。


## 15. 重新載入，驗證第二份 Colab 所需的資料

從剛才保存的檔案重新載入，檢查完整性、文件編號與向量對應，再使用**同一個問題向量**比對保存前後的排名與分數。此處不重算問題向量，以免不同運算裝置的細微浮點誤差干擾索引保存檢查。

也將 FAISS 結果與 NumPy 的全矩陣內積排序比較，確認精確搜尋的意義。遇到完全同分時，兩套工具的同分排列順序可以不同。

In [ ]:
loaded_config = json.loads((EXPORT_DIR / "config.json").read_text(encoding="utf-8"))
for name, expected_hash in loaded_config["file_sha256"].items():
    actual_hash = hashlib.sha256((EXPORT_DIR / name).read_bytes()).hexdigest()
    assert actual_hash == expected_hash, f"檔案完整性檢查失敗：{name}"

loaded_index = faiss.read_index(str(EXPORT_DIR / "index.faiss"))
loaded_documents = json.loads((EXPORT_DIR / "documents.json").read_text(encoding="utf-8"))
loaded_ids = json.loads((EXPORT_DIR / "index_doc_ids.json").read_text(encoding="utf-8"))
assert loaded_index.ntotal == len(loaded_documents) == len(loaded_ids) == loaded_config["document_count"]
assert loaded_index.d == loaded_config["dimension"]
assert len(set(loaded_ids)) == len(loaded_ids)
assert set(loaded_ids) == {doc["doc_id"] for doc in loaded_documents}
assert loaded_documents == documents and loaded_ids == index_doc_ids

verification_query = embed_question(QUESTION)
k = min(TOP_K, index.ntotal)
before_scores, before_positions = index.search(verification_query, k)
after_scores, after_positions = loaded_index.search(verification_query, k)
np.testing.assert_array_equal(before_positions, after_positions)
np.testing.assert_allclose(before_scores, after_scores, atol=1e-6, rtol=1e-6)

all_scores = (embeddings @ verification_query[0]).ravel()
expected_top_scores = np.sort(all_scores)[::-1][:k]
np.testing.assert_allclose(after_scores[0], expected_top_scores, atol=1e-5, rtol=1e-5)
np.testing.assert_allclose(after_scores[0], all_scores[after_positions[0]], atol=1e-5, rtol=1e-5)
print("通過：檔案完整性、文件對照、索引維度、保存前後一致，以及精確搜尋檢查。")

通過：檔案完整性、文件對照、索引維度、保存前後一致，以及精確搜尋檢查。


## 16. 打包並下載知識庫

ZIP 根目錄包含前述四個檔案。下載後，留給第二份 Colab 上傳使用；模型權重不在 ZIP 裡。


In [ ]:
EXPORT_ZIP = EXPORT_DIR.parent / f"{EXPORT_DIR.name}_download" / EXPORT_ZIP_NAME
EXPORT_ZIP.parent.mkdir(parents=True, exist_ok=True)
required_files = ["index.faiss", "documents.json", "index_doc_ids.json", "config.json"]
with zipfile.ZipFile(EXPORT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for name in required_files:
        archive.write(EXPORT_DIR / name, arcname=name)
with zipfile.ZipFile(EXPORT_ZIP) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(required_files)
print("下載檔名：", EXPORT_ZIP_NAME)
print("大小：", f"{EXPORT_ZIP.stat().st_size:,} bytes")
files.download(str(EXPORT_ZIP))

下載檔名： vector_db.zip
大小： 44,988 bytes


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>